# Clase 081 — Regularización de árboles

Un árbol sin regularizar crece hasta hojas puras → *overfitting* garantizado. Controlamos la
complejidad con **pre-pruning** (`max_depth`, `min_samples_leaf`, `max_leaf_nodes`) y con
**post-pruning** (`ccp_alpha`, *cost-complexity pruning*), eligiendo los hiperparámetros con
**curvas de validación**, no a ojo.

Requiere: `numpy`, `matplotlib`, `scikit-learn`.

## 1. Overfit baseline

Un `DecisionTreeClassifier()` sin tocar nada sobre `make_moons(noise=0.4)` memoriza el train:
*accuracy* de train ≈ 1 y una brecha grande contra el test.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

RND = 42
X, y = make_moons(n_samples=1000, noise=0.4, random_state=RND)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RND)

base = DecisionTreeClassifier(random_state=RND).fit(X_train, y_train)
acc_tr = accuracy_score(y_train, base.predict(X_train))
acc_te = accuracy_score(y_test, base.predict(X_test))
print(f"baseline: acc_train={acc_tr:.3f}  acc_test={acc_te:.3f}")
print(f"gap train-test: {acc_tr - acc_te:.3f}")
assert acc_tr > acc_te, "el arbol sin regularizar overfittea"
print("train casi perfecto y brecha grande = overfitting")

## 2. Sweep de max_depth

Barremos `max_depth` y graficamos *accuracy* de train y test. El punto donde train sigue
subiendo pero test se estanca o cae marca el inicio del *overfitting*.

In [ ]:
depths = [1, 2, 3, 5, 10, 20]
tr_scores, te_scores = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=RND).fit(X_train, y_train)
    tr_scores.append(accuracy_score(y_train, m.predict(X_train)))
    te_scores.append(accuracy_score(y_test, m.predict(X_test)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(depths, tr_scores, "o-", label="train")
ax.plot(depths, te_scores, "s-", label="test")
ax.set_xlabel("max_depth"); ax.set_ylabel("accuracy")
ax.set_title("Overfitting al crecer max_depth"); ax.legend()
plt.tight_layout(); plt.show()
print("mejor test acc en depth =", depths[int(np.argmax(te_scores))])

## 3. GridSearchCV (pre-pruning)

Buscamos con validación cruzada la mejor combinación de `max_depth`, `min_samples_leaf` y
`max_leaf_nodes`.

In [ ]:
param_grid = {
    "max_depth": [3, 5, 7, None],
    "min_samples_leaf": [1, 5, 20],
    "max_leaf_nodes": [8, 16, None],
}
grid = GridSearchCV(DecisionTreeClassifier(random_state=RND),
                    param_grid, cv=5, n_jobs=1)
grid.fit(X_train, y_train)
print("mejores params:", grid.best_params_)
acc_grid = accuracy_score(y_test, grid.predict(X_test))
print(f"acc_test (grid): {acc_grid:.3f}")

## 4. Cost-complexity pruning (post-pruning)

`cost_complexity_pruning_path` devuelve la secuencia de `ccp_alphas`. Entrenamos un árbol por
cada `α`, graficamos test *accuracy* vs. `α` y elegimos el óptimo **con la curva**.

In [ ]:
path = DecisionTreeClassifier(random_state=RND).cost_complexity_pruning_path(
    X_train, y_train)
alphas = path.ccp_alphas[:-1]        # descartamos el alpha que deja solo la raiz
accs = []
for a in alphas:
    m = DecisionTreeClassifier(random_state=RND, ccp_alpha=a).fit(X_train, y_train)
    accs.append(accuracy_score(y_test, m.predict(X_test)))
accs = np.array(accs)
best_alpha = alphas[int(np.argmax(accs))]
print(f"mejor ccp_alpha: {best_alpha:.5f}  ->  acc_test={accs.max():.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alphas, accs, marker=".", drawstyle="steps-post")
ax.axvline(best_alpha, color="r", ls="--", label=f"mejor alpha={best_alpha:.4f}")
ax.set_xlabel("ccp_alpha"); ax.set_ylabel("accuracy test")
ax.set_title("Cost-complexity pruning"); ax.legend()
plt.tight_layout(); plt.show()

## 5. Comparación de fronteras

Visualizamos la frontera del árbol **sin regularizar** (ondulada, memoriza ruido) contra la de
un árbol con `max_depth=4` (suave, generaliza).

In [ ]:
def plot_boundary(ax, model, X, y, title):
    x0 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 250)
    x1 = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 250)
    xx, yy = np.meshgrid(x0, x1)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=12)
    ax.set_title(title)

reg = DecisionTreeClassifier(max_depth=4, random_state=RND).fit(X_train, y_train)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_boundary(axes[0], base, X_test, y_test, "Sin regularizar (overfit)")
plot_boundary(axes[1], reg, X_test, y_test, "max_depth=4 (regularizado)")
plt.tight_layout(); plt.show()

## Ejercicios

1. Entrená un `DecisionTreeClassifier()` sin regularizar sobre `make_moons(noise=0.4)` y
   reportá el gap train/test.
2. Barré `max_depth ∈ {1, 2, 3, 5, 10, None}` y graficá train y test *accuracy*. Identificá el
   punto donde empieza el *overfit*.
3. Buscá con `GridSearchCV(cv=5)` la mejor combinación de `max_depth`, `min_samples_leaf` y
   `max_leaf_nodes`.
4. Obtené los `ccp_alphas` con `cost_complexity_pruning_path`, entrená un árbol por cada `α` y
   elegí el óptimo justificándolo con la curva test *accuracy* vs. `α`.
5. Graficá la frontera del árbol sin regularizar vs. `max_depth=4`.

## Conclusiones

- Un árbol sin regularizar crece hasta hojas puras → *overfitting* casi garantizado.
- **Pre-pruning** (`max_depth`, `min_samples_leaf`, `max_leaf_nodes`) es rápido y cubre el 90%
  de los casos.
- **Post-pruning** (`ccp_alpha`) es el método con mejor fundamento teórico; se elige con
  `cost_complexity_pruning_path` + CV.
- Las decisiones deben justificarse con **curvas de validación**, no a ojo.
- `min_samples_split=2` (default) casi no frena nada; para regularizar realmente hay que
  subirlo o usar `min_samples_leaf`.